# COVID-19 Transmission, incubation & environmental stability

This is a summary Notebook of key findings for this Task. It builds upon the Notebook [Thematic tagging with Regular Expressions](https://www.kaggle.com/ajrwhite/covid-19-thematic-tagging-with-regular-expressions/) and `covid19_tools` utility script.

### Contents

- [Reproduction rate ($R$ / $R_0$)](#Reproduction)
- [Incubation period](#Incubation)
- [Persistence / Environmental stability](#Persistence)

In [1]:
import covid19_tools as cv19
import pandas as pd
import re
from IPython.core.display import display, HTML

METADATA_FILE = '../input/CORD-19-research-challenge/metadata.csv'

# Load metadata
meta = cv19.load_metadata(METADATA_FILE)
# Add tags
meta, covid19_counts = cv19.add_tag_covid19(meta)
meta, transmission_counts = cv19.count_and_tag(meta,
                                               cv19.TRANSMISSION_SYNONYMS,
                                               'transmission_generic')
meta, repr_counts = cv19.count_and_tag(meta,
                                       cv19.REPR_SYNONYMS,
                                       'transmission_repr')
meta, incubation_counts = cv19.count_and_tag(meta,
                                             cv19.INCUBATION_SYNONYMS,
                                             'transmission_incub')
meta, persistence_counts = cv19.count_and_tag(meta,
                                              cv19.PERSISTENCE_SYNONYMS,
                                              'transmission_persist')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

loaded DataFrame with 59887 records
Added tag_disease_covid19 to DataFrame
Added tag_transmission_generic to DataFrame
Added tag_transmission_repr to DataFrame
Added tag_transmission_incub to DataFrame
Added tag_transmission_persist to DataFrame


In [2]:
print('Loading full text for tag_disease_covid19')
full_text_repr = cv19.load_full_text(meta[meta.tag_disease_covid19 &
                                          meta.tag_transmission_repr],
                                     '../input/CORD-19-research-challenge')

Loading full text for tag_disease_covid19
Found 120 full texts for 176 records


In [3]:
table_repr = []
repr_strings = [r'\br0\b', r'$r_0$', r'\br 0\b',
                'reproduction number', 'reproduction rate',
                'reproductive number', 'rate of reproduction']

for i, record in enumerate(full_text_repr):
    sha = record['paper_id']
    meta_row = meta[meta.sha == sha]
    temp_dict = {
        'publication_date': meta_row.publish_time.values[0],
        'authors': meta_row.authors_short.values[0],
        'doi': meta_row.doi.values[0],
        'title': meta_row.title.values[0],
        'journal': meta_row.journal.values[0],
        'key_passages': []
    }

    for item in record['body_text']:
        if 'value of' in item['text'].lower() or 'estimated' in item['text'].lower():
            sentences = item['text'].split('. ')
            for s in sentences:
                if len(re.findall('|'.join(repr_strings), s.lower())) > 0:
                    if len(re.findall(r'\d+\.\d+', s)) > 0:
                        temp_dict['key_passages'].append(s)
    if len(temp_dict['key_passages']) == 0:
        temp_dict['key_passages'] = ['<i>Failed to extract figures - check manually.</i>']
    table_repr.append(temp_dict)
    
table_repr = pd.DataFrame(table_repr)

table_repr['key_passages'] = (table_repr
                              .key_passages
                              .apply(lambda x: '<br><br>'.join(x)))
table_repr['title_link'] = table_repr.apply(lambda x: f'<a href="{x.doi}">{x.title}</a> ({x.journal})',
                                                 axis=1)
# table_repr.drop(['title', 'doi'], axis=1, inplace=True)

# Reproduction

In [4]:
cv19.display_dataframe(table_repr[['publication_date',
                                   'authors',
                                   'title_link',
                                   'key_passages']],
                       'Table of Reproduction Rates (<i>R</i> / <i>R<sub>0</sub></i>)')

publication_date,authors,title_link,key_passages
2020-04-29,Mohd Hafiz Mohd et al,Unraveling the Myths of R0 in Controlling the Dynamics of COVID-19 Outbreak: a Modelling Perspective (nan),Failed to extract figures - check manually.
2020-04-29,William Marciel de Souza et al,Epidemiological and clinical characteristics of the early phase of the COVID-19 epidemic in Brazil (nan),Failed to extract figures - check manually.
2020-04-29,Ines Abdeljaoued-Tej,COVID-19 data analysis and modeling in Palestine (nan),"For these values of parameters, R 0 = 1.54In the literature, the value of R 0 varies from 1.4 to 3.9 [8] We find that minimum value of R 0 is equal to 1.18 (f = 0.7 and 1/ν = 3), the maximum is equal to 3.3 (f = 0.1 and 1/ν = 8) and the median is equal to 2.24Figure 9 : In this figure, we use 1/η = 7 days, and we plot the basic reproductive number R 0 as a function of f and 1/ν using τ = 4.55 10 −6 , which corresponds to the data for Palestine in Table 4 Figure 10: Number of infected cases and deaths"
2020-04-29,Conghui Xu et al,Forecast analysis of the epidemics trend of COVID-19 in the United States by a generalized fractional-order SEIR model (nan),Failed to extract figures - check manually.
2020-04-29,Anatoly Zhigljavsky et al,A prototype for decision support tool to help decision-makers with the strategy of handling the COVID-19 UK epidemic (nan),"Authors of [6] suggest R 0 = 2.2 and R 0 = 2.4 as typical; the authors of [8] use values for R 0 in the range [2.25, 2.75]We shall use the value R 0 = 2.5 as typical, which, in view of the recent data, looks to be a rather high (pessimistic) choice overallHowever, R 0 = 2.5 seems to be an adequate choice for the mega-cities where the epidemics develop faster and may lead to more causalitiesUK's experts believe r 0.009 [6] , WHO sets the world-wide mortality rate at 0.034, the authors of [14] believe r is very small and could be close to 0.001, an Israeli expert DIn view of recent data, the value of R 0 is likely to be lower than 2.5, especially in small towns and rural areasFor London, Sheffield and Manchester the reproductive number R 0 could be close to 2.5 but for rural areas and small town R 0 should be much smaller, even close to 1"
2020-04-29,Zhenzhen Lu et al,A fractional-order SEIHDR model for COVID-19 with inter-city networked coupling effects (nan),Failed to extract figures - check manually.
2020-04-27,You Chen et al,Modeling COVID-19 Growing Trends to Reveal the Differences in the Effectiveness of Non-Pharmaceutical Interventions among Countries in the World (nan),"For instance, it requires 19 days to reach 33 cases per million population at a reproduction rate Rd as 1.3, while 39 days to reach the same number of cases at a reproduction rate as 1.2."
2020-04-26,Li-Shan Huang et al,Taking Account of Asymptomatic Infections in Modeling the Transmission Potential of the COVID-19 Outbreak on the Diamond Princess Cruise Ship (nan),"The R0 of COVID-19 on DP has been estimated previously in [15] ; they identified the R0 as 14.8 initially and then declining to a stable 1.78 after the quarantine and removal interventionsOther researchers using data on the DP up to February 16 have estimated the median R0 as 2.28 [16] R0 for passengers in (d)(iv) is 4.73 (95% CI (4.37, 5.12)) and 4.39 (95%CI (4.06, 4.75)) at t = 4, 5, respectively, and 1.06 (95%CI (0.98, 1.15)) and 1.05 (95%CI (0.97, 1.14)) for crew in (d)(ii) respectivelyThis shows that R0 for some passengers increased from 3.78 to 4.73 from t = 3 to t = 4 if they were in contact with asymptomatic crew membersFrom t = 4 to t = 5, R0 for those passengers decreased slightly to 4.39, mostly due to removal of a larger number of infected passengersWhen τ = 5, rp = 0.2, aratio = 0.505, and estimated β = 3.27, the R0 for passengers in (d)(iv) is 4.18 (95% CI (3.86, 4.52)), 4.08 (95%CI (3.77, 4.42)), and 3.74 (95% CI (3.45, 4.04)) at t = 4, 5, and 6 respectively, and for crew in (d)(ii) is 0.92 (95%CI (0.85, 1.00)), 0.92 (95%CI (0.8

# Incubation

In [5]:
print('Loading full text for tag_transmission_incub')
full_text_incub = cv19.load_full_text(meta[meta.tag_disease_covid19 &
                                           meta.tag_transmission_incub],
                                      '../input/CORD-19-research-challenge')

Loading full text for tag_transmission_incub
Found 101 full texts for 168 records


In [6]:
table_incub = []
time_strings = []
for unit in ['day', 'hour', 'hr']:
    time_strings += [f'\\d+ {unit}', f'\\d+\\.\\d+ {unit}', f'{unit} \\d']

for i, record in enumerate(full_text_incub):
    sha = record['paper_id']
    meta_row = meta[meta.sha == sha]
    temp_dict = {
        'publication_date': meta_row.publish_time.values[0],
        'authors': meta_row.authors_short.values[0],
        'doi': meta_row.doi.values[0],
        'title': meta_row.title.values[0],
        'journal': meta_row.journal.values[0],
        'key_passages': []
    }

    for item in record['body_text']:
        if 'value of' in item['text'].lower() or 'estimated' in item['text'].lower():
            sentences = item['text'].split('. ')
            for s in sentences:
                if len(re.findall('|'.join(cv19.INCUBATION_SYNONYMS), s.lower())) > 0:
                    if len(re.findall('|'.join(time_strings), s.lower())) > 0:
                        temp_dict['key_passages'].append(s)
    if len(temp_dict['key_passages']) == 0:
        temp_dict['key_passages'] = ['<i>Failed to extract figures - check manually.</i>']
    table_incub.append(temp_dict)
    
table_incub = pd.DataFrame(table_incub)

table_incub['key_passages'] = (table_incub
                                 .key_passages
                                 .apply(lambda x: '<br><br>'.join(x)))
table_incub['title_link'] = table_incub.apply(lambda x: f'<a href="{x.doi}">{x.title}</a> ({x.journal})',
                                                    axis=1)
# table_incub.drop(['title', 'doi'], axis=1, inplace=True)

In [7]:
cv19.display_dataframe(table_incub[['publication_date',
                                    'authors',
                                    'title_link',
                                    'key_passages']],
                       'Table of Incubation Periods')

publication_date,authors,title_link,key_passages
2020-06-30,Xia et al,Epidemiological and initial clinical characteristics of patients with family aggregation of COVID-19 (Journal of Clinical Virology),Failed to extract figures - check manually.
2020-05-31,Han et al,COVID-19 in a patient with long-term use of glucocorticoids: A study of a familial cluster (Clinical Immunology),Failed to extract figures - check manually.
2020-04-30,Tian et al,Characteristics of COVID-19 infection in Beijing (Journal of Infection),Failed to extract figures - check manually.
2020-04-29,Lei et al,Author's reply - Clinical characteristics and outcomes of patients undergoing surgeries during the incubation period of COVID-19 infection (EClinicalMedicine),Failed to extract figures - check manually.
2020-04-29,Sohail et al,Forecasting the timeframe of coronavirus and human cells interaction with reverse engineering (Progress in Biophysics and Molecular Biology),Failed to extract figures - check manually.
2020-04-29,D Pal et al,Mathematical Analysis of a COVID-19 Epidemic Model by using Data Driven Epidemiological Parameters of Diseases Spread in India (nan),Failed to extract figures - check manually.
2020-04-29,pongkaew udomsamuthirun et al,The reproductive index from SEIR model of Covid-19 epidemic in Asean (nan),Failed to extract figures - check manually.
2020-04-28,Liu et al,A COVID-19 epidemic model with latency period (Infectious Disease Modelling),Failed to extract figures - check manually.
2020-04-28,Andrew C Berry et al,Wisconsin April 2020 Election Not Associated with Increase in COVID-19 Infection Rates (nan),Failed to extract figures - check manually.
2020-04-27,Cimolai,"More data are required for incubation period, infectivity, and quarantine duration for COVID-19 (Travel Medicine and Infectious Disease)",Failed to extract figures - check manually.


# Persistence

In [8]:
print('Loading full text for tag_transmission_persist')
full_text_persist = cv19.load_full_text(meta[meta.tag_disease_covid19 &
                                             meta.tag_transmission_persist],
                                             '../input/CORD-19-research-challenge')

Loading full text for tag_transmission_persist
Found 96 full texts for 124 records


In [9]:
table_persist = []
time_strings = []
for unit in ['day', 'minute', 'min', 'hour', 'hr', 'second', 'sec']:
    time_strings += [f'\\d+ {unit}', f'\\d+\\.\\d+ {unit}', f'{unit} \\d']

for i, record in enumerate(full_text_persist):
    sha = record['paper_id']
    meta_row = meta[meta.sha == sha]
    temp_dict = {
        'publication_date': meta_row.publish_time.values[0],
        'authors': meta_row.authors_short.values[0],
        'doi': meta_row.doi.values[0],
        'title': meta_row.title.values[0],
        'journal': meta_row.journal.values[0],
        'key_passages': []
    }

    for item in record['body_text']:
        sentences = item['text'].split('. ')
        for s in sentences:
            if len(re.findall('|'.join(cv19.PERSISTENCE_SYNONYMS), s.lower())) > 0:
#                 if len(re.findall('|'.join(time_strings), s.lower())) > 0:
                temp_dict['key_passages'].append(s)
    if len(temp_dict['key_passages']) == 0:
        temp_dict['key_passages'] = ['<i>Failed to extract figures - check manually.</i>']
    table_persist.append(temp_dict)
    
table_persist = pd.DataFrame(table_persist)

table_persist['key_passages'] = (table_persist
                                 .key_passages
                                 .apply(lambda x: '<br><br>'.join(x)))
table_persist['title_link'] = table_persist.apply(lambda x: f'<a href="{x.doi}">{x.title}</a> ({x.journal})',
                                                    axis=1)
# table_persist.drop(['title', 'doi'], axis=1, inplace=True)

In [10]:
cv19.display_dataframe(table_persist.loc[table_persist.key_passages
                                         != '<i>Failed to extract figures - check manually.</i>',
                                     ['publication_date',
                                      'authors',
                                      'title_link',
                                      'key_passages']],
                       'Table of Persistence Findings')

publication_date,authors,title_link,key_passages
2020-08-31,Jacobi et al,Portable chest X-ray in coronavirus disease-19 (COVID-19): A pictorial review (Clinical Imaging),"However, due to infection control issues related to patient transport to CT suites, the inefficiencies introduced in CT room decontamination, and lack of CT availability in parts of the world, portable chest radiography (CXR) will likely be the most commonly utilized modality for identification and follow up of lung abnormalitiesIn fact, the American College of Radiology (ACR) notes that CT decontamination required after scanning COVID-19 patients may disrupt radiological service availability and suggests that portable chest radiography may be considered to minimize the risk of cross-infection (American College of Radiology [3] )"
2020-08-10,Heller et al,COVID-19 faecal-oral transmission: Are we asking the right questions? (Science of The Total Environment),"c o m / l o c a t e / s c i t o t e n v persistence of viable SARS-Cov-2 in water and sewage has yet to be determinedPersistence of human coronaviruses on surfaces is highly variable (from 2 h to 9 days), depending on temperature, humidity, type of surface, and virus strain (Kampf et al., 2020) However, a comprehensive and more nuanced analysis is required to test this hypothesis, taking into consideration both environmental dynamics and the persistence of viral infectivity.Based on the environmental routes of excreted diseases, and the state-of-knowledge of SARS-CoV-2 persistence and infectiousness, FigAdditionally, a fifth category emerges: a ""water-cleaning"" category, in which contaminated water used to clean surfaces may, via hand contact, bring the virus to the mouthValidation of this framework will require significant research efforts to better understand SARS-CoV-2 persistence and infectivity in stools, sewage, and untreated water; the role of vectors in transporting the virus, and the appropriate investigation of the ""water-cleaning"" route."
2020-05-31,Wang et al,SARS-CoV-2 RNA detection of hospital isolation wards hygiene monitoring during the Coronavirus Disease 2019 outbreak in a Chinese hospital (International Journal of Infectious Diseases),"The occurrence of healthcare-associated infections might be closely related to pathogens contamination in the hospital environment (Beggs et al., 2015 , Chowell et al., 2015 In the past, several studies on environmental sampling have performed to identify the contamination in field-settings with SARS or MERS-CoV (Bin et al., 2016; Otter et al., 2016) Three zones including contaminated area, semi-contaminated area, and clean areaA contaminated area is a specifically designated area for patients of COVID-19 and contaminated items such as patients' wastesA clean area is a specifically designated area for non-contaminated itemsSemi-contaminated area was set up between the contaminated area and the clean areaBuffer room is set up between a contaminated zone and a semi-contaminated zoneHospital environmental sampling and staff personal protective equipment (PPE) sites sampling During February 19th-24 th , 2020, we collected samples of the environmental surfaces in Isolation ICU ward and Isolation wards, including the clean area, the semi-contaminated area, and the contaminated areaEnvironmental surfaces were sampled by using ClassiqSwabs (Copan Flock Technologies, Brescia, Italy), and collected in universal transport medium (UTM) containing Hanks' Balanced Salt Solution, BSA, HEPES, amino acids, glycerin and so onDetection of SARS-CoV-2 RNA among health-care settings, sewage, and staffs' PPE In routine cleaning and disinfection, the 36 samples of environmental surface in isolation wards including the clean area, the semi-contaminated area, and the contaminated area were all negativeIn recent studies, environmental sampling also has identified contamination in field-settings with SARS-CoV-2 With routine cleaning and disinfection, none of SARS-CoV-2 RNA was d

In [11]:
meta.to_csv('augmented_metadata_full.csv', index=False)

In [12]:
table_repr.to_csv('reproduction_table.csv', index=False)

In [13]:
table_incub.to_csv('incubation_table.csv', index=False)

In [14]:
table_persist.to_csv('persistence_table.csv', index=False)